# Treinamento Baseline: VGG16 (Transfer Learning)

## Objetivo
Este notebook estabelece o **modelo base (baseline)** para o projeto de Reconhecimento de Expressões Faciais (FER). O objetivo é treinar uma CNN estática adaptada e avaliar seu desempenho inicial antes de avançarmos para arquiteturas temporais (RNNs/LSTMs) ou modelos baseados em atenção, conforme o escopo da pesquisa.

## Dados e Pré-processamento
* **Dataset:** FER2013 (*In-the-wild*).
* **Imagens:** Redimensionadas para 48x48 pixels.
* **Conversão:** Transformadas para RGB e pré-processadas com a função nativa `vgg16.preprocess_input` para adequação à rede pré-treinada.

## Arquitetura da Rede
Utilizamos a técnica de *Transfer Learning* (congelando as camadas convolucionais iniciais) para extração de características, evitando o *overfitting* em dados limitados:
1. **Base:** VGG16 (pesos do *ImageNet*).
2. **Classificador Customizado:**
   * `Flatten`
   * `Dense` (256 neurônios, ativação *ReLU*)
   * `Dropout` (0.5)
   * `Dense` (7 neurônios, ativação *Softmax*)

## Salvamento Automático
O notebook registra as métricas de acurácia e *loss* em um arquivo `.csv` a cada época e salva o melhor modelo treinado `.keras` para inferências futuras e construção dos gráficos comparativos.

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications import vgg16
from tensorflow.keras.callbacks import CSVLogger, ModelCheckpoint # NOVO: Importando os callbacks

# Definindo os caminhos e parâmetros básicos
caminho_treino = './fer2013/train'
caminho_teste = './fer2013/test'
tamanho_lote = 32 # Quantas imagens a IA vê por vez
tamanho_imagem = (48, 48) # Tamanho padrão do FER2013

print("Carregando dados de treinamento...")
dados_treino = tf.keras.utils.image_dataset_from_directory(
    caminho_treino,
    color_mode="rgb", # A VGG16 exige imagens com 3 canais de cor (RGB)
    image_size=tamanho_imagem,
    batch_size=tamanho_lote
)

print("\nCarregando dados de validação (teste)...")
dados_teste = tf.keras.utils.image_dataset_from_directory(
    caminho_teste,
    color_mode="rgb",
    image_size=tamanho_imagem,
    batch_size=tamanho_lote
)

# Aplica o pré-processamento exato que a VGG16 exige
dados_treino = dados_treino.map(lambda x, y: (vgg16.preprocess_input(x), y))
dados_teste = dados_teste.map(lambda x, y: (vgg16.preprocess_input(x), y))

# Carrega a rede pré-treinada sem o "topo" (include_top=False)
base_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(48, 48, 3))

# Congela os pesos para não estragar o que ela já aprendeu
base_vgg.trainable = False

# Construindo o novo "topo" da rede
modelo_vgg = models.Sequential([
    base_vgg,           # A base da VGG16 (extrator de características)
    layers.Flatten(),   # Transforma os mapas de características 2D em um vetor 1D
    layers.Dense(256, activation='relu'), # Camada densa para processar os padrões extraídos
    layers.Dropout(0.5), # Ajuda a prevenir o overfitting durante o treinamento
    layers.Dense(7, activation='softmax')  # Saída final para as 7 classes de emoções do FER2013
])

# Compilando o modelo
modelo_vgg.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

modelo_vgg.summary()

# ---------------------------------------------------------
# NOVO: Configurando os Salvamentos Automáticos (Callbacks)
# ---------------------------------------------------------
csv_logger_vgg = CSVLogger('historico_vgg16_fer2013.csv', append=True)

checkpoint_vgg = ModelCheckpoint(
    'melhor_modelo_vgg16.keras', 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max',
    verbose=1
)

# Treinamento com os Callbacks ativados (aumentei para 30 épocas para um treino real)
print("\nIniciando o treinamento com VGG16... Acompanhe a evolução da acurácia.")
historico = modelo_vgg.fit(
    dados_treino,
    validation_data=dados_teste,
    epochs=30, 
    callbacks=[csv_logger_vgg, checkpoint_vgg] # <-- Callbacks injetados!
)

Carregando dados de treinamento...
Found 28709 files belonging to 7 classes.

Carregando dados de validação (teste)...
Found 7178 files belonging to 7 classes.
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, 1, 1, 512)         14714688  
                                                                 
 flatten (Flatten)           (None, 512)               0         
                                                                 
 dense (Dense)               (None, 256)               131328    
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 7)                 1799      
                                                                 
Total params: 14,847,815
Tra

# Treinamento Comparativo: ResNet50 (Baseline com Data Augmentation)

## Objetivo
Expandir a análise de arquiteturas de redes convolucionais profundas (CNNs) aplicando a **ResNet50**. O intuito desta etapa é avaliar como uma rede baseada em blocos residuais (*skip connections*) performa na classificação de microexpressões estáticas em relação à arquitetura sequencial da VGG16, mantendo o uso restrito de *Transfer Learning* puro.

## Pré-processamento e Aumento de Dados
* **Pré-processamento Específico:** Os pixels passam por um recálculo nativo do Keras (`resnet50.preprocess_input`) adequado especificamente para o que a ResNet espera receber.
* **Data Augmentation:** Para combater o viés e o desbalanceamento severo de classes do dataset *in-the-wild* (FER2013), implementamos uma camada de aumento de dados *on-the-fly* contendo espelhamento horizontal, rotação aleatória (10%) e zoom (10%).

## Arquitetura da Rede
Para garantir uma comparação justa e controlada contra a VGG16, o classificador do "topo" foi mantido idêntico:
1. **Base:** ResNet50 (pesos do *ImageNet* totalmente **congelados**).
2. **Classificador Customizado:**
   * `Flatten`
   * `Dense` (256 neurônios, ativação *ReLU*)
   * `Dropout` (0.5)
   * `Dense` (7 neurônios, ativação *Softmax*)

## Registro de Experimento
As métricas evolutivas e o modelo mais otimizado são salvos de forma independente (`historico_resnet50_fer2013.csv` e `melhor_modelo_resnet50.keras`) para alimentar os gráficos da etapa de Discussão Crítica e Análise Comparativa de Arquiteturas.

In [5]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications import resnet50 # Pré-processamento específico da ResNet
from tensorflow.keras.callbacks import CSVLogger, ModelCheckpoint

# 0. É essencial aplicar o pré-processamento específico da ResNet50 também!
dados_treino_resnet = dados_treino.map(lambda x, y: (resnet50.preprocess_input(x), y))
dados_teste_resnet = dados_teste.map(lambda x, y: (resnet50.preprocess_input(x), y))

# 1. Camada de Data Augmentation para ajudar no desbalanceamento
data_augmentation = models.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
])

# 2. Base ResNet-50
base_resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(48, 48, 3))
base_resnet.trainable = False

# 3. Modelo Completo
modelo_resnet = models.Sequential([
    data_augmentation, # Aplica o aumento de dados antes de tudo
    base_resnet,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

modelo_resnet.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# ---------------------------------------------------------
# Callbacks Exclusivos para a ResNet50
# ---------------------------------------------------------
csv_logger_resnet = CSVLogger('historico_resnet50_fer2013.csv', append=True)

checkpoint_resnet = ModelCheckpoint(
    'melhor_modelo_resnet50.keras', 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max',
    verbose=1
)

# 4. Treino
print("Iniciando ResNet-50 com Data Augmentation...")
historico_resnet = modelo_resnet.fit(
    dados_treino_resnet, 
    validation_data=dados_teste_resnet, 
    epochs=30,
    callbacks=[csv_logger_resnet, checkpoint_resnet] # <-- Callbacks da ResNet injetados!
)

Iniciando ResNet-50 com Data Augmentation...
Epoch 1/30
897/898 [============================>.] - ETA: 0s - loss: 1.9282 - accuracy: 0.2664
Epoch 1: val_accuracy improved from -inf to 0.30858, saving model to melhor_modelo_resnet50.keras
898/898 [==============================] - 116s 125ms/step - loss: 1.9281 - accuracy: 0.2665 - val_loss: 1.6804 - val_accuracy: 0.3086
Epoch 2/30
897/898 [============================>.] - ETA: 0s - loss: 1.7437 - accuracy: 0.2860
Epoch 2: val_accuracy improved from 0.30858 to 0.37197, saving model to melhor_modelo_resnet50.keras
898/898 [==============================] - 110s 122ms/step - loss: 1.7436 - accuracy: 0.2860 - val_loss: 1.6477 - val_accuracy: 0.3720
Epoch 3/30
897/898 [============================>.] - ETA: 0s - loss: 1.7135 - accuracy: 0.3000
Epoch 3: val_accuracy did not improve from 0.37197
898/898 [==============================] - 109s 122ms/step - loss: 1.7136 - accuracy: 0.3000 - val_loss: 1.6183 - val_accuracy: 0.3571
Epoch 4/30
8

# Estratégia A: ResNet50 com Resampling Espacial (UpSampling)

## Objetivo
A arquitetura ResNet50 foi originalmente projetada para processar imagens maiores (como 224x224 pixels). Quando utilizamos as imagens nativas de 48x48 do FER2013, a rede reduz tanto a dimensionalidade nas sucessivas camadas de *pooling* que acaba perdendo as características espaciais cruciais do rosto humano (como os vincos da testa ou a curva da boca). Esta estratégia busca avaliar se o aumento artificial da imagem destrava o limite de acurácia do modelo.

## Modificações na Arquitetura
* **Lupa Digital (`UpSampling2D`):** Adicionada como a primeira camada do modelo para esticar a imagem de `48x48` para `96x96` antes de entregá-la à ResNet.
* **Data Augmentation:** Implementação de espelhamento horizontal e rotações leves para atenuar o desbalanceamento das classes do dataset.
* **Base Neural:** Pesos da ResNet50 mantidos 100% congelados (*Transfer Learning* clássico).

## Registro de Experimento
Geração isolada dos arquivos `historico_resnet50_upsampling.csv` e `melhor_modelo_resnet50_upsampling.keras` para posterior cruzamento de dados na análise comparativa.

In [3]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import CSVLogger, ModelCheckpoint

print("Montando ResNet50 com UpSampling2D (96x96)...")

# 1. Base ResNet-50 (Atenção ao input_shape alterado para 96x96)
base_resnet_up = ResNet50(weights='imagenet', include_top=False, input_shape=(96, 96, 3))
base_resnet_up.trainable = False

# 2. Modelo Completo com a "Lupa Digital"
modelo_resnet_up = models.Sequential([
    layers.UpSampling2D(size=(2, 2), input_shape=(48, 48, 3)), # <--- A mágica acontece aqui (48 vira 96)
    layers.RandomFlip("horizontal"), # Data Augmentation leve
    layers.RandomRotation(0.1),
    base_resnet_up,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

modelo_resnet_up.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 3. Callbacks Exclusivos
csv_logger_up = CSVLogger('historico_resnet50_upsampling.csv', append=True)
checkpoint_up = ModelCheckpoint(
    'melhor_modelo_resnet50_upsampling.keras', 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max',
    verbose=1
)

# 4. Treino
historico_up = modelo_resnet_up.fit(
    dados_treino_resnet, 
    validation_data=dados_teste_resnet, 
    epochs=30,
    callbacks=[csv_logger_up, checkpoint_up]
)

Montando ResNet50 com UpSampling2D (96x96)...
Epoch 1/30
898/898 [==============================] - ETA: 0s - loss: 1.9594 - accuracy: 0.2771
Epoch 1: val_accuracy improved from -inf to 0.34313, saving model to melhor_modelo_resnet50_upsampling.keras
898/898 [==============================] - 221s 243ms/step - loss: 1.9594 - accuracy: 0.2771 - val_loss: 1.6296 - val_accuracy: 0.3431
Epoch 2/30
898/898 [==============================] - ETA: 0s - loss: 1.7013 - accuracy: 0.2988
Epoch 2: val_accuracy improved from 0.34313 to 0.41794, saving model to melhor_modelo_resnet50_upsampling.keras
898/898 [==============================] - 241s 268ms/step - loss: 1.7013 - accuracy: 0.2988 - val_loss: 1.5628 - val_accuracy: 0.4179
Epoch 3/30
898/898 [==============================] - ETA: 0s - loss: 1.6500 - accuracy: 0.3349
Epoch 3: val_accuracy improved from 0.41794 to 0.43327, saving model to melhor_modelo_resnet50_upsampling.keras
898/898 [==============================] - 225s 250ms/step - lo

# Estratégia B: ResNet50 com Fine-Tuning Parcial

## Objetivo
Na abordagem padrão, a base da ResNet50 estava totalmente congelada. Isso forçava a rede a classificar emoções faciais utilizando exatamente os mesmos padrões genéricos extraídos do *ImageNet* (focado em identificar animais, carros, objetos, etc.). O objetivo desta etapa é permitir que a arquitetura refine seus conhecimentos, adaptando os filtros mais profundos especificamente para a identificação de microexpressões humanas.

## Modificações na Arquitetura
* **Descongelamento Gradual:** A base da ResNet50 foi liberada, mas bloqueamos todas as camadas exceto as **15 últimas**. Isso preserva a detecção de bordas e formas básicas nas camadas iniciais, enquanto treina a semântica facial nas camadas finais.
* **Micro-ajustes (`Learning Rate` reduzida):** Otimizador configurado com taxa de aprendizado extremamente baixa (`1e-5`). Um valor padrão destruiria os pesos pré-treinados rapidamente; a taxa baixa garante um aprendizado suave e adaptativo.
* **Dimensionalidade:** Mantida em `48x48` pixels nativos.

## Registro de Experimento
Geração isolada dos arquivos `historico_resnet50_finetuning.csv` e `melhor_modelo_resnet50_finetuning.keras` para posterior cruzamento de dados na análise comparativa.

In [4]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import CSVLogger, ModelCheckpoint
import tensorflow as tf

print("Montando ResNet50 com Fine-Tuning (Descongelando últimas camadas)...")

# 1. Base ResNet-50 original
base_resnet_ft = ResNet50(weights='imagenet', include_top=False, input_shape=(48, 48, 3))

# 2. O Fine-Tuning: Descongelamos as últimas 15 camadas da rede
base_resnet_ft.trainable = True
for layer in base_resnet_ft.layers[:-15]:
    layer.trainable = False

# 3. Modelo Completo
modelo_resnet_ft = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    base_resnet_ft,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

# ATENÇÃO: No Fine-Tuning, a taxa de aprendizado (learning rate) deve ser MUITO BAIXA (ex: 1e-5).
# Se for alta, os pesos pré-treinados são destruídos rapidamente.
otimizador_lento = tf.keras.optimizers.Adam(learning_rate=1e-5)

modelo_resnet_ft.compile(optimizer=otimizador_lento, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 4. Callbacks Exclusivos
csv_logger_ft = CSVLogger('historico_resnet50_finetuning.csv', append=True)
checkpoint_ft = ModelCheckpoint(
    'melhor_modelo_resnet50_finetuning.keras', 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max',
    verbose=1
)

# 5. Treino
historico_ft = modelo_resnet_ft.fit(
    dados_treino_resnet, 
    validation_data=dados_teste_resnet, 
    epochs=30,
    callbacks=[csv_logger_ft, checkpoint_ft]
)

Montando ResNet50 com Fine-Tuning (Descongelando últimas camadas)...
Epoch 1/30
897/898 [============================>.] - ETA: 0s - loss: 2.3530 - accuracy: 0.2507
Epoch 1: val_accuracy improved from -inf to 0.34271, saving model to melhor_modelo_resnet50_finetuning.keras
898/898 [==============================] - 167s 183ms/step - loss: 2.3530 - accuracy: 0.2506 - val_loss: 1.6975 - val_accuracy: 0.3427
Epoch 2/30
898/898 [==============================] - ETA: 0s - loss: 1.7752 - accuracy: 0.3109
Epoch 2: val_accuracy improved from 0.34271 to 0.37127, saving model to melhor_modelo_resnet50_finetuning.keras
898/898 [==============================] - 169s 188ms/step - loss: 1.7752 - accuracy: 0.3109 - val_loss: 1.6141 - val_accuracy: 0.3713
Epoch 3/30
897/898 [============================>.] - ETA: 0s - loss: 1.6848 - accuracy: 0.3401
Epoch 3: val_accuracy improved from 0.37127 to 0.39440, saving model to melhor_modelo_resnet50_finetuning.keras
898/898 [==============================]

# Treinamento Comparativo: EfficientNetB0 (Eficiência e Escala)

## Objetivo
Para fechar a tríade de arquiteturas CNN estáticas com *Transfer Learning*, este notebook introduz a **EfficientNetB0**. Diferente da VGG16 (focada em profundidade sequencial) e da ResNet50 (focada em blocos residuais), a família EfficientNet utiliza um redimensionamento composto (*compound scaling*) que equilibra largura, profundidade e resolução da rede de forma matemática. O objetivo é testar se essa arquitetura otimizada consegue extrair características do FER2013 com mais precisão.

## Dados e Pré-processamento
* **Pré-processamento Específico:** Transformação para RGB e aplicação do mapa de escalonamento nativo da arquitetura (`efficientnet.preprocess_input`).
* **Data Augmentation:** Aplicação de espelhamento horizontal e rotação aleatória (10%) diretamente no pipeline do modelo, forçando a rede a aprender de forma mais generalista para lidar com o desbalanceamento.

## Arquitetura da Rede
Este modelo adota uma abordagem mais moderna para a transição entre a extração de características e a classificação:
1. **Base:** EfficientNetB0 (pesos do *ImageNet* totalmente **congelados**).
2. **Classificador Customizado:**
   * `GlobalAveragePooling2D` (Substitui o `Flatten` tradicional, compactando os dados de forma mais inteligente e reduzindo drasticamente as chances de *overfitting* espacial).
   * `Dense` (256 neurônios, ativação *ReLU*).
   * `Dropout` (0.5).
   * `Dense` (7 neurônios para as classes de emoções, ativação *Softmax*).

## Registro de Experimento (Treinamento Extenso)
Como a EfficientNet possui uma dinâmica de aprendizado diferente, o treinamento foi configurado para **50 épocas**. O acompanhamento será gravado no `historico_efficientnetb0_fer2013.csv`, enquanto o melhor conjunto de pesos será blindado no arquivo `melhor_modelo_efficientnetb0.keras`.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import efficientnet
from tensorflow.keras.callbacks import CSVLogger, ModelCheckpoint

# 1. Parâmetros e Caminhos
caminho_treino = './fer2013/train'
caminho_teste = './fer2013/test'
tamanho_lote = 32
tamanho_imagem = (48, 48) 

print("Carregando dados...")
dados_treino = tf.keras.utils.image_dataset_from_directory(caminho_treino, color_mode="rgb", image_size=tamanho_imagem, batch_size=tamanho_lote)
dados_teste = tf.keras.utils.image_dataset_from_directory(caminho_teste, color_mode="rgb", image_size=tamanho_imagem, batch_size=tamanho_lote)

# 2. Pré-processamento e Augmentation
dados_treino = dados_treino.map(lambda x, y: (efficientnet.preprocess_input(x), y))
dados_teste = dados_teste.map(lambda x, y: (efficientnet.preprocess_input(x), y))

data_augmentation = models.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
])

# 3. Base EfficientNetB0 (Congelada)
base_eff = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(48, 48, 3))
base_eff.trainable = False

# 4. Arquitetura
modelo_eff = models.Sequential([
    data_augmentation,
    base_eff,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

modelo_eff.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# ---------------------------------------------------------
# 5. Callbacks de Salvamento (CORRIGIDO)
# ---------------------------------------------------------
csv_logger = CSVLogger('historico_efficientnetb0_fer2013.csv', append=True)
checkpoint = ModelCheckpoint(
    'melhor_pesos_efficientnetb0.h5', # Mudei para .h5 (mais estável para pesos)
    monitor='val_accuracy', 
    save_best_only=True, 
    save_weights_only=True, # <--- ISSO RESOLVE O BUG! Salva só o "cérebro", não a estrutura.
    mode='max',
    verbose=1
)

# 6. Treino
print("\nIniciando EfficientNetB0. Agora sim, bom descanso!")
historico = modelo_eff.fit(
    dados_treino,
    validation_data=dados_teste,
    epochs=30,
    callbacks=[csv_logger, checkpoint]
)

Carregando dados...
Found 28709 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.

Iniciando EfficientNetB0. Agora sim, bom descanso!
Epoch 1/30
898/898 [==============================] - ETA: 0s - loss: 1.6716 - accuracy: 0.3333
Epoch 1: val_accuracy improved from -inf to 0.38799, saving model to melhor_pesos_efficientnetb0.h5
898/898 [==============================] - 55s 56ms/step - loss: 1.6716 - accuracy: 0.3333 - val_loss: 1.5616 - val_accuracy: 0.3880
Epoch 2/30
897/898 [============================>.] - ETA: 0s - loss: 1.6166 - accuracy: 0.3620
Epoch 2: val_accuracy improved from 0.38799 to 0.39621, saving model to melhor_pesos_efficientnetb0.h5
898/898 [==============================] - 49s 55ms/step - loss: 1.6167 - accuracy: 0.3620 - val_loss: 1.5343 - val_accuracy: 0.3962
Epoch 3/30
898/898 [==============================] - ETA: 0s - loss: 1.6055 - accuracy: 0.3682
Epoch 3: val_accuracy did not improve from 0.39621
898/898 [============================

# Experimento Avançado: SMOTE em Deep Features (VGG16)

## Objetivo
A aplicação direta do SMOTE em pixels gerou distorções espaciais (*ghosting*) que invalidaram o aprendizado da CNN. Esta abordagem avança para a aplicação do SMOTE no espaço latente. Utilizamos a rede VGG16 como um extrator de características para traduzir as imagens em vetores semânticos abstratos. O SMOTE é então aplicado exclusivamente sobre esses vetores unidimensionais, criando representações lógicas perfeitamente balanceadas de todas as classes sem comprometer a estrutura visual.

## Pipeline Metodológico
1. **Extração de Características:** Passagem do dataset inteiro pela VGG16 (sem o topo), gerando tensores unidimensionais através da camada `GlobalAveragePooling2D`.
2. **Balanceamento Latente:** O algoritmo SMOTE equilibra as classes preenchendo as lacunas geométricas no espaço das *Deep Features*.
3. **Classificação Top-Level:** Treinamento de uma rede Densa simples (Multilayer Perceptron) focada unicamente em classificar os vetores balanceados.

In [2]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, vgg16
from tensorflow.keras.callbacks import CSVLogger, ModelCheckpoint
from imblearn.over_sampling import SMOTE

caminho_treino = './fer2013/train'
caminho_teste = './fer2013/test'
tamanho_imagem = (48, 48)

print("Carregando dados em RGB para a VGG16...")
dados_treino = tf.keras.utils.image_dataset_from_directory(caminho_treino, color_mode="rgb", image_size=tamanho_imagem, batch_size=32)
dados_teste = tf.keras.utils.image_dataset_from_directory(caminho_teste, color_mode="rgb", image_size=tamanho_imagem, batch_size=32)

# Extraindo todas as imagens para a memória e aplicando o pré-processamento da VGG16
x_treino_lista, y_treino_lista = [], []
for imagens, rotulos in dados_treino:
    x_treino_lista.append(vgg16.preprocess_input(imagens.numpy()))
    y_treino_lista.append(rotulos.numpy())

X_treino = np.concatenate(x_treino_lista, axis=0)
y_treino = np.concatenate(y_treino_lista, axis=0)

print("\nCarregando a VGG16 para extração de características...")
# O pooling='avg' já transforma o resultado final num vetor 1D, economizando o Flatten
base_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(48, 48, 3), pooling='avg')

print("Processando o dataset através da VGG16 (Isso leva um tempinho)...")
features_treino = base_vgg.predict(X_treino, batch_size=32)

print("\nAplicando SMOTE no espaço latente das características...")
smote = SMOTE(random_state=42)
features_balanceadas, y_treino_balanceado = smote.fit_resample(features_treino, y_treino)

print(f"Dataset original: {features_treino.shape} | Dataset balanceado: {features_balanceadas.shape}")

# Construindo a rede final (O "cérebro" decisor)
modelo_topo = models.Sequential([
    layers.Dense(256, activation='relu', input_shape=(features_balanceadas.shape[1],)),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

modelo_topo.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

csv_logger = CSVLogger('historico_vgg16_smote.csv', append=True)
checkpoint = ModelCheckpoint('melhor_topo_vgg16_smote.keras', monitor='val_accuracy', save_best_only=True, mode='max')

print("\nTreinando o classificador final nas Deep Features sintéticas...")
# Para validação, precisamos extrair as features do conjunto de teste também!
x_teste_lista, y_teste_lista = [], []
for imagens, rotulos in dados_teste:
    x_teste_lista.append(vgg16.preprocess_input(imagens.numpy()))
    y_teste_lista.append(rotulos.numpy())

X_teste = np.concatenate(x_teste_lista, axis=0)
y_teste = np.concatenate(y_teste_lista, axis=0)
features_teste = base_vgg.predict(X_teste, batch_size=32)

historico = modelo_topo.fit(
    features_balanceadas, y_treino_balanceado,
    validation_data=(features_teste, y_teste),
    epochs=30,
    batch_size=32,
    callbacks=[csv_logger, checkpoint]
)

Carregando dados em RGB para a VGG16...
Found 28709 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.

Carregando a VGG16 para extração de características...
Processando o dataset através da VGG16 (Isso leva um tempinho)...
898/898 [==============================] - 83s 92ms/step

Aplicando SMOTE no espaço latente das características...
Dataset original: (28709, 512) | Dataset balanceado: (50505, 512)

Treinando o classificador final nas Deep Features sintéticas...
225/225 [==============================] - 21s 94ms/step
Epoch 1/30
1579/1579 [==============================] - 3s 2ms/step - loss: 2.4550 - accuracy: 0.2977 - val_loss: 1.6679 - val_accuracy: 0.3430
Epoch 2/30
1579/1579 [==============================] - 3s 2ms/step - loss: 1.6213 - accuracy: 0.3758 - val_loss: 1.6064 - val_accuracy: 0.3753
Epoch 3/30
1579/1579 [==============================] - 3s 2ms/step - loss: 1.5460 - accuracy: 0.4036 - val_loss: 1.6029 - val_accuracy: 0.3766
Epoch 4/30
1579/1579